# Group Project - Safe routes home at night 

**DS105A Group Project – Data for Data Science (Autumn Term 2025/2026)**

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #ED9255; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

**Group Notebook**
- 👤 Team Name: CtrlAltElite
- 🎯 Purpose: This notebook contains code outlining our data collection approach. 
</div>

In [7]:
import requests
import json
import os
import pandas as pd
from pathlib import Path
import shelve

## 1. Gather nightclub data for a popular area in London

This [TimeOut list ](https://www.timeout.com/london/clubs/the-best-clubs-in-london) features 5 clubs in Dalston, the most for any area on the list. This test will use Dalston to generate clubs.

In [14]:
# Constants
places_url = 'https://places.googleapis.com/v1/places:searchText'
API_KEY = os.getenv('API_KEY')

In [3]:
# Request google places text search for nightclubs in Dalston

headers = {
    'Content-Type': 'application/json',
    'X-Goog-API-Key': API_KEY,
    'X-Goog-FieldMask': '*',
    # IMPORTANT: using '*' for FieldMask returns all fields. I am only using this for the test - don't use when we know what fields we want returned because it could increase costs.
}

data = {
    'textQuery': 'nightclubs in Dalston, London, England'
}

response = requests.post(url=places_url, headers=headers, data=json.dumps(data))

if response.status_code == 200:
    print('Success!')
else:
    print(f'Error: {response.status_code}: {response.text}')

Success!


### Explore API response

In [7]:
print(len(response.json()))
print(response.json().keys())

4
dict_keys(['places', 'contextualContents', 'nextPageToken', 'searchUri'])


### Store response locally

In [ ]:
data=response.json()
with open('data/raw/dalson_nightclubs.json', 'w') as f:
    json.dump(data, f, indent=2)

### Explore `dalston_nightclubs.json`

In [3]:
with open('data/raw/dalston_nightclubs.json', 'r') as f:
    data = json.load(f)

In [29]:
data.keys()

dict_keys(['places', 'contextualContents', 'nextPageToken', 'searchUri'])

In [27]:
data['places'][0].keys()

dict_keys(['name', 'id', 'types', 'nationalPhoneNumber', 'internationalPhoneNumber', 'formattedAddress', 'addressComponents', 'plusCode', 'location', 'viewport', 'rating', 'googleMapsUri', 'websiteUri', 'regularOpeningHours', 'utcOffsetMinutes', 'adrFormatAddress', 'businessStatus', 'priceLevel', 'userRatingCount', 'iconMaskBaseUri', 'iconBackgroundColor', 'displayName', 'primaryTypeDisplayName', 'takeout', 'delivery', 'dineIn', 'curbsidePickup', 'reservable', 'servesBeer', 'servesWine', 'currentOpeningHours', 'primaryType', 'shortFormattedAddress', 'editorialSummary', 'reviews', 'photos', 'outdoorSeating', 'liveMusic', 'servesCocktails', 'goodForChildren', 'restroom', 'goodForGroups', 'goodForWatchingSports', 'paymentOptions', 'accessibilityOptions', 'addressDescriptor', 'googleMapsLinks', 'priceRange', 'reviewSummary', 'timeZone', 'postalAddress'])

In [28]:
# check to see if all clubs have the same data

data['places'][0].keys() == data['places'][1].keys()

False

In [5]:
dalston_clubs = data['places']

In [ ]:
# filter to relevant columns if present

## desired_cols = [
##        'businessStatus', gvuy
##        'displayName',
##        'currentOpeningHours',
##        'regularOpeningHours',
##        'primaryType',
##        'name',
##        'id',
##        'types',
##        'formattedAddress',
##        'location'
##    ]

## 2. Gather nearest station data for each club

In [15]:
# Split latitude and longitude into separate strings for the first club in dalston_clubs

dalston_clubs[0]['location']
first_club_lat, first_club_long = (
    dalston_clubs[0]['location']['latitude'],
    dalston_clubs[0]['location']['longitude']
)

In [35]:
# Test constants

API_KEY = os.getenv('API_KEY')
STATIONS_URL = 'https://places.googleapis.com/v1/places:searchNearby'
HEADERS = {
        'Content-Type': 'application/json',
        'X-Goog-API-Key': API_KEY,
        'X-Goog-FieldMask': 'places.displayName,places.location,places.primaryType',
    }
ROUNDING = 4 # round location data to avoid unneccesary API calls for similar locations - this allows for roughly a 10m distance 

In [ ]:
# Test request

data = {
        'includedTypes' : ['bus_station', 'subway_station', 'train_station'],
        'maxResultCount' : 1,
        'rankPreference': 'DISTANCE',
        'locationRestriction' : {
            'circle': {
                'center' : {
                    'latitude' : first_club_lat,
                    'longitude' : first_club_long
                },
                'radius' : 400
            }
        }
    }
        

response = requests.post(url=STATIONS_URL, headers=HEADERS, json=data)

if response.status_code != 200:
    print(f'Error: {response.status_code}: {response.text}')

In [40]:
# Inspect API response

print(dalston_clubs[0]['displayName'])
print('-' * 50)
response.json()

{'text': 'Dalston Superstore', 'languageCode': 'en'}
--------------------------------------------------


{'places': [{'location': {'latitude': 51.5500602, 'longitude': -0.0750841},
   'displayName': {'text': 'Rio Cinema (Stop T)', 'languageCode': 'en'},
   'primaryType': 'bus_station'}]}

## 3. Write a function to automate the nearest station data collection

In [32]:
# Constants for google places nearby API

API_KEY = os.getenv('API_KEY')
STATIONS_URL = 'https://places.googleapis.com/v1/places:searchNearby'
HEADERS = {
        'Content-Type': 'application/json',
        'X-Goog-API-Key': API_KEY,
        'X-Goog-FieldMask': 'places.displayName,places.location,places.primaryType',
    }
CACHE_DIR = Path('/files/group_project/cache_dir')
CACHE_FILE = CACHE_DIR / 'station_cache_v1.json'
ROUNDING = 4 # round location data to avoid unneccesary API calls for similar locations - this allows for roughly a 10m distance 

In [11]:
# check if cache file exists / load cache file

if CACHE_FILE.exists():
    station_cache = json.loads(CACHE_FILE.read_text())
else:
    station_cache = {}

In [33]:
# Define a function to request google places nearby API for nearest public transport stations for a location

def get_nearest_station(lat, lon):

    cache_key = f'{round(lat, ROUNDING)},{round(lon, ROUNDING)}'

    if cache_key in station_cache:
        return station_cache[cache_key]
    # avoid API calls if location already used

    data = {
        'includedTypes' : ['bus_station', 'subway_station', 'train_station'],
        'maxResultCount' : 1,
        'rankPreference': 'DISTANCE',
        'locationBias' : {
            'point': {
                'latitude' : lat,
                'longitude' : lon
            }
        }
    }
        

    response = requests.post(url=STATIONS_URL, headers=HEADERS, json=data)

    if response.status_code != 200:
        print(f'Error: {response.status_code}: {response.text}')
        return None

    places = response.json().get('places', [])
        
    if places:
        station = {
            'name': places[0]['displayName']['text'],
            'latitude': places[0]['location']['latitude'],
            'longitude': places[0]['location']['longitude']
        }  
            
        station_cache[cache_key] = station
        print(json.dumps(response.json(), indent=2))
        return station

    # no station found at any radius
    station_cache[cache_key] = None
    return None

In [34]:
# test  nearest station API collection with first dalston clubs

test_station = get_nearest_station(first_club_lat, first_club_long)
print(test_station)

None


### Police API Limitations

_Data Quality_

The [Police API](https://data.police.uk/about/) lists some known issues regarding data quality. A particularly important issue for our purposes is that there are inconsistent geocoding policies in police forces. Estimates of geocoding accuracy in different forces range from 60% to 97%. We cannot filter for confident location data so we will acknowledge this limitation in our report.